# M3 step 6: screenshot simulation (task 2, RQ2 normal edit)

A screenshot changes a photo in small ways: it is resized to the screen, slightly blurred, and its
colours shift a little. This notebook checks whether the stamp survives that.

Each of the 500 test photos is stamped with its fixed verdict and confidence, then:

1. resized to 80% of its size,
2. blurred with a Gaussian of radius 0.5,
3. brightness and contrast each changed by a fixed random factor between -5% and +5%,
4. resized back to the original size,
5. saved as PNG.

Each step is also tested on its own, to see which part does most of the damage. One extra row adds
JPEG quality 90 after the screenshot, since screenshots are often shared as JPEG.

Expected: the stamp should mostly survive, since resizing and mild changes were part of training.
Blur was not trained and may cost some bits.

**GPU:** not needed. About 5 to 10 minutes on a CPU.

## Setup

**Kaggle:** turn on *Settings > Accelerator > GPU T4 x2* (only notebooks 8, 9 and 11 need it) and
*Internet on*, and attach the `veristamp-m3-inputs` dataset (README.md, "Running on Kaggle"). The first
cell then finds the code, installs what is missing and moves into `m3_experiments`. No path to edit.

**Colab or a local machine:** set the paths once in `settings.py`, then run from the `m3_experiments`
folder.

In [ ]:
# Kaggle setup: find the dataset wherever Kaggle put it, copy the code, point the notebooks at the files.
import os, shutil, subprocess, sys
from pathlib import Path

def find(rel, max_depth=6):
    for depth in range(max_depth + 1):
        hits = sorted(Path('/kaggle/input').glob('/'.join(['*'] * depth + [rel])))
        if hits:
            return hits[0]
    return None

if Path('/kaggle/input').is_dir() and not Path('settings.py').exists():
    code = find('m3_experiments/settings.py')
    model = find('m3/best.pt')
    print('Code found at: ', code)
    print('Model found at:', model)
    if code is None or model is None:
        print(subprocess.run(['find', '/kaggle/input', '-maxdepth', '4'], capture_output=True, text=True).stdout[:3000])
        raise SystemExit('The veristamp-m3-inputs dataset is not attached, or has a different layout (see the list above).')
    repo = Path('/kaggle/working/VeriStamp')
    if not (repo / 'm3_experiments' / 'settings.py').exists():
        shutil.rmtree(repo, ignore_errors=True)          # removes a failed earlier clone
        shutil.copytree(code.parent.parent, repo, ignore=shutil.ignore_patterns('.git'))
    # Each item is found on its own, so the inputs can come in several zips or datasets.
    paths = {'M3E_MODEL_PATH': 'm3/best.pt', 'M3E_TEST_DIR': 'm3/test', 'M3E_TEST_MANIFEST': 'm3/test_manifest.csv',
             'M3E_GENIMAGE_SPLIT_DIR': 'genimage/splits', 'M3E_GENIMAGE_ROOT': 'genimage/images',
             'M3E_M1_CHECKPOINT': 'm1/best.pt', 'M3E_M1_VSLIB_DIR': 'm1/vslib_m1_final'}
    for name, rel in paths.items():
        hit = find(rel)
        os.environ[name] = str(hit if hit else model.parent.parent / rel)   # the notebooks run from here inherit these
        print(f'  {rel:<22} {"found" if hit else "not in the inputs"}')
    os.environ['M3E_WORK_DIR'] = '/kaggle/working/m3_work'
    os.chdir(repo / 'm3_experiments')
    for pkg in ('diffusers', 'lpips'):
        try:
            __import__(pkg)
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('Working in', Path.cwd())

In [ ]:
# On Colab instead, uncomment and run once per session:
# from google.colab import drive; drive.mount('/content/drive')
# !git clone https://github.com/KruthikaVirupakshappa/VeriStamp.git /content/VeriStamp
# %cd /content/VeriStamp/m3_experiments
# !pip install -q diffusers transformers accelerate scikit-image lpips

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt

import settings as S
import m3_eval as E
mw = S.setup()                      # imports ../m3_watermark/m3_watermark.py as it is
torch.manual_seed(S.SEED)
np.random.seed(S.SEED)

m3 = mw.load_m3(S.MODEL_PATH)       # load once, reuse for every photo
DEVICE = m3.device
print('Model epoch', m3.epoch, 'on', DEVICE)

In [ ]:
files = E.list_photos(S.TEST_DIR, S.N_TEST_PHOTOS)
if len(files) != S.N_TEST_PHOTOS:
    raise RuntimeError(f'Expected {S.N_TEST_PHOTOS} test photos in {S.TEST_DIR}, found {len(files)}')
# Every photo gets a fixed verdict, P(AI) and tag (E.payload_for), the same in every notebook.
# The stamped PNGs are made once and reused by notebooks 6 to 9.
records = E.stamp_test_set(mw, m3, files, S.WORK_DIR, S.MODEL_PATH, S.SEED)
print(len(records), 'stamped photos in', S.WORK_DIR / 'stamped')
records.head(3)

## Run every edit on the stamped photos

In [ ]:
per_photo = pd.concat([E.run_edit(mw, m3, records, edit, name) for name, edit in E.SCREENSHOT_EDITS.items()],
                      ignore_index=True)
summary = E.summarise(per_photo)
E.as_percent(summary)

## Photos without a stamp

The same screenshot simulation on the unstamped photos. None of them should be read as having a valid
stamp.

In [ ]:
unstamped = E.run_edit(mw, m3, records, E.SCREENSHOT_EDITS['Screenshot simulation (all steps)'],
                       'Screenshot simulation (all steps)', source='original')
false_valid = E.false_valid_summary(unstamped)
false_valid

## One example

In [ ]:
r = records.iloc[0]
original, stamped = E.open_rgb(r['original']), E.open_rgb(r['stamped'])
shot = E.screenshot(stamped, r['name'])
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, im, title in zip(axes, [original, stamped, shot], ['Original', 'Stamped', 'After screenshot simulation']):
    ax.imshow(im); ax.set_title(title); ax.axis('off')
plt.tight_layout()
(S.RESULTS_DIR / 'figures').mkdir(exist_ok=True)
plt.savefig(S.RESULTS_DIR / 'figures' / 'rq2_screenshot_example.png', dpi=110, bbox_inches='tight')
plt.show()
print(E.score_read(mw, m3, shot, r['expected_bits'], r['verdict']))

## Save

In [ ]:
E.save_results(S.RESULTS_DIR, 'rq2_screenshot', per_photo=pd.concat([per_photo, unstamped]), summary=summary,
               meta={'false_valid_unstamped': false_valid.to_dict('records'), 'seed': S.SEED,
                     'steps': 'resize 0.8 bilinear -> GaussianBlur(0.5) -> brightness, contrast x U(0.95,1.05) '
                              '-> resize back bilinear -> PNG',
                     'environment': E.environment_info()})
print(E.to_markdown(E.as_percent(summary)))

## What the results say

Write two or three sentences here once the table is in: did the stamp survive the full screenshot
simulation, and which single step cost the most bits? Compare with the expected result above.